# 06 - Deep Error Analysis

Detailed error analysis of the DenseNet-Attention model:
- Prediction confidence distribution
- False positive & false negative analysis
- Grad-CAM visualizations of critical errors
- Clinical threshold sensitivity analysis

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from src.data import get_dataloaders
from src.models import get_model
from src.evaluate import generate_gradcam, plot_gradcam_grid, compute_metrics
from src.utils import set_seed, get_device, load_config

plt.rcParams["figure.dpi"] = 130
sns.set_style("whitegrid")

config = load_config("../configs/default.yaml")
device = get_device()
DATA_ROOT = "../data/chest_xray"
SEED = config["reproducibility"]["seed"]

print(f"Device: {device}")

In [ ]:
set_seed(SEED)

dataloaders = get_dataloaders(
    DATA_ROOT,
    augmentation="standard",
    image_size=config["data"]["image_size"],
    batch_size=32,
    num_workers=0,
    seed=SEED
)

model = get_model("densenet_attention", pretrained=True, use_attention=True)
checkpoint_path = "../results/checkpoints/densenet_attention_best.pt"

model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
model = model.to(device)
model.eval()

print("✅ Best DenseNet-Attention model loaded.")

In [ ]:
test_images = []
test_labels = []
test_probs = []

with torch.no_grad():
    for images, labels in dataloaders["test"]:
        outputs = model(images.to(device)).squeeze()
        probs = torch.sigmoid(outputs).cpu().numpy()
        
        test_images.append(images)
        test_labels.append(labels.numpy())
        test_probs.extend(probs)

test_images = torch.cat(test_images)
test_labels = np.concatenate(test_labels)
test_probs = np.array(test_probs)
test_preds = (test_probs >= 0.5).astype(int)

print(f"Test set processed: {len(test_labels)} images")

In [ ]:
bins = [0.0, 0.5, 0.7, 0.85, 0.95, 0.99, 1.0]
bin_labels = pd.cut(test_probs, bins=bins, 
                    labels=[f"{bins[i]:.2f}–{bins[i+1]:.2f}" for i in range(len(bins)-1)])
correct = test_labels == test_preds

print("=== CONFIDENCE DISTRIBUTION vs ERROR RATE ===\n")
for b in bin_labels.categories:
    mask = bin_labels == b
    if mask.sum() > 0:
        error_rate = (~correct[mask]).mean()
        print(f"Confidence {b:>12s} → Error rate: {error_rate:.1%}  ({mask.sum()} samples)")

In [ ]:
fp_idx = np.where((test_preds == 1) & (test_labels == 0))[0]
fn_idx = np.where((test_preds == 0) & (test_labels == 1))[0]

print(f"False Positives : {len(fp_idx)} images ({len(fp_idx)/len(test_labels)*100:.1f}%)")
print(f"False Negatives: {len(fn_idx)} images ({len(fn_idx)/len(test_labels)*100:.1f}%)")

high_conf_fp = fp_idx[np.argsort(test_probs[fp_idx])[-8:]]
high_conf_fn = fn_idx[np.argsort(test_probs[fn_idx])[:8]]

In [ ]:
target_layer = model.features.denseblock4.denselayer16.conv2

print("Generating Grad-CAM for high-confidence False Positives...")
heatmaps_fp = generate_gradcam(model, test_images[high_conf_fp], target_layer, device)
fig_fp = plot_gradcam_grid(test_images[high_conf_fp], heatmaps_fp, test_labels[high_conf_fp], test_probs[high_conf_fp], n_samples=8, title="High-Confidence False Positives")
plt.show()

print("Generating Grad-CAM for False Negatives...")
heatmaps_fn = generate_gradcam(model, test_images[high_conf_fn], target_layer, device)
fig_fn = plot_gradcam_grid(test_images[high_conf_fn], heatmaps_fn, test_labels[high_conf_fn], test_probs[high_conf_fn], n_samples=8, title="False Negatives (Missed Pneumonia)")
plt.show()

In [ ]:
print("=== THRESHOLD SENSITIVITY ANALYSIS (Clinical Perspective) ===\n")
thresholds = [0.3, 0.5, 0.7, 0.85, 0.95]

for t in thresholds:
    metrics = compute_metrics(test_labels, test_probs, threshold=t)
    print(f"Threshold {t:.2f} | Sens: {metrics['sensitivity']:.3f} | "
          f"Spec: {metrics['specificity']:.3f} | F1: {metrics['f1_macro']:.3f} | NPV: {metrics['npv']:.3f}")

## Summary & Clinical Recommendations

- Model shows high confidence even on errors
- False negatives are particularly critical in screening
- Lower threshold increases sensitivity (ideal for triage)
- Higher threshold improves specificity
- **Next steps**: Ensemble + Test-Time Augmentation